# Cyber Threat Forecasting: Vision Model Verification

This notebook verifies the VisionTS pipeline for cyber threat forecasting. It mirrors the structure of `Graph_Pipeline_Verification.ipynb`.

## Environment & Path Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# --- Add Project Root to Path ---
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

# Add VisionTS submodule path
visionts_path = os.path.join(project_root, 'Transformers', 'Visual_Transformer')
if visionts_path not in sys.path:
    sys.path.insert(0, visionts_path)

print(f"Project Root added to path: {project_root}")

# Import custom modules
from Transformer_Pipeline.Cyber_Trend_Vision_Config import CyberVisionTSConfig
from Transformer_Pipeline.Preprocessing.Load_Data import load_cyber_threat_data
from Transformer_Pipeline.Preprocessing.Cyber_Trend_to_Image import (
    apply_double_exponential_smoothing,
    create_sliding_windows,
    split_windowed_data,
    compute_reference_scaler
)
from Transformer_Pipeline.Cyber_Trend_Image_Dataset import CyberTrendImageDataset
from Transformer_Pipeline.Models.VisionTS_Wrapper import VisionTSModel, create_visionts_model

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Setup Complete. Running on: {device}")

## Configuration & Constants

Defines the experiment hyperparameters aligned with B-MTGNN benchmark settings.

In [ ]:
# --- B-MTGNN Benchmark Settings ---
CONSTANTS = {
    'TRAIN_SPLIT': 0.43,
    'VAL_SPLIT': 0.30,
    'CONTEXT_LEN': 10,       # 10 Months History
    'PRED_LEN': 36,          # 3 Years Future (36 Months)
    'BATCH_SIZE': 1,         # Use 1 for memory efficiency with 1231 features
    'OUTPUT_DIR': os.path.join(project_root, 'Processed_Data', 'vision_notebook_test')
}

# VisionTS-specific settings
VISIONTS_CONFIG = {
    'model_arch': 'mae_base',
    'finetune_type': 'ln',
    'periodicity': 1,        # No seasonality for monthly cyber data
    'norm_const': 0.4,
    'align_const': 0.4,
    'interpolation': 'bilinear',
    'ckpt_dir': os.path.join(project_root, 'Processed_Data', 'vision', 'ckpt'),
    'load_pretrained': True,
}

# Ensure output directory exists
os.makedirs(CONSTANTS['OUTPUT_DIR'], exist_ok=True)
os.makedirs(VISIONTS_CONFIG['ckpt_dir'], exist_ok=True)

print("Experimental Settings:")
print(f" - Input Window (context_len): {CONSTANTS['CONTEXT_LEN']} Months")
print(f" - Forecast Horizon (pred_len): {CONSTANTS['PRED_LEN']} Months")
print(f" - Batch Size: {CONSTANTS['BATCH_SIZE']} (reduced for memory with high-dim data)")
print(f" - VisionTS Architecture: {VISIONTS_CONFIG['model_arch']}")

## Data Ingestion & Verification

Loads the raw dataset and verifies the time-series structure.

In [ ]:
raw_data_path = os.path.join(project_root, 'Data_Preparation', 'Cyber_Trend_Forecasting_All.csv')

# Load Data
df_raw = load_cyber_threat_data(raw_data_path)

if df_raw is not None:
    print("\n--- Data Verification ---")
    print(f"Shape: {df_raw.shape} (months x threat_types)")
    print(f"Index is monotonic increasing? {df_raw.index.is_monotonic_increasing}")
    print(f"Start Date: {df_raw.index.min().date()}")
    print(f"End Date:   {df_raw.index.max().date()}")
    print(f"\nFirst 5 columns: {list(df_raw.columns[:5])}")

    display(df_raw.head(3))
else:
    print("Failed to load data.")

## Double Exponential Smoothing (Visual Check)

Applies DES to reduce noise - same preprocessing as the Graph pipeline.

In [ ]:
# Apply Double Exponential Smoothing
df_smooth = apply_double_exponential_smoothing(df_raw, alpha=0.1, beta=0.1)

# Plot comparison for a specific threat
target_col = df_raw.columns[0]  # First column
if target_col in df_raw.columns:
    plt.figure(figsize=(12, 5))
    plt.plot(df_raw.index, df_raw[target_col], label='Raw', alpha=0.5, color='gray')
    plt.plot(df_smooth.index, df_smooth[target_col], label='Smoothed (DES)', color='blue', linewidth=2)
    plt.title(f"Effect of Double Exponential Smoothing on '{target_col}'")
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Data Windowing & Splitting

Create sliding windows on the full dataset, then split into train/val/test.

**Key Differences from Graph Pipeline**:
1. VisionTS uses 3D tensors `(samples, time, features)` instead of 4D
2. We "window first, then split" to ensure all splits have valid samples with small datasets
3. External normalization is NOT applied (VisionTS handles normalization internally)

In [ ]:
print("--- Data Preparation ---")

# 1. Compute reference scaler on training portion (NOT applied - VisionTS handles normalization internally)
n = len(df_smooth)
train_end = int(n * CONSTANTS['TRAIN_SPLIT'])
train_portion = df_smooth.iloc[:train_end]
scaler = compute_reference_scaler(train_portion, CONSTANTS['OUTPUT_DIR'])

# 2. Create sliding windows on FULL dataset (window first, then split)
print("\n--- Creating Sliding Windows on Full Dataset ---")
X_all, y_all = create_sliding_windows(df_smooth, CONSTANTS['CONTEXT_LEN'], CONSTANTS['PRED_LEN'])

# 3. Split windowed data into train/val/test
X_train, y_train, X_val, y_val, X_test, y_test = split_windowed_data(
    X_all, y_all, CONSTANTS['TRAIN_SPLIT'], CONSTANTS['VAL_SPLIT']
)

# Save for Dataset class
np.savez(os.path.join(CONSTANTS['OUTPUT_DIR'], 'train.npz'), x=X_train, y=y_train)
np.savez(os.path.join(CONSTANTS['OUTPUT_DIR'], 'val.npz'), x=X_val, y=y_val)
np.savez(os.path.join(CONSTANTS['OUTPUT_DIR'], 'test.npz'), x=X_test, y=y_test)

print(f"\nTensor Shapes (VisionTS expects 3D):")
print(f"X (Input):  {X_train.shape} -> (Samples, context_len={CONSTANTS['CONTEXT_LEN']}, nvars={X_train.shape[2]})")
print(f"y (Target): {y_train.shape} -> (Samples, pred_len={CONSTANTS['PRED_LEN']}, nvars={y_train.shape[2]})")
print(f"\nTotal samples: {X_all.shape[0]} -> Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

## Dataset & DataLoader Setup

Initialize the `CyberTrendImageDataset` and create PyTorch DataLoaders.

In [ ]:
# Initialize Datasets (loading from preprocessed files)
train_dataset = CyberTrendImageDataset(data_dir=CONSTANTS['OUTPUT_DIR'], split='train')
val_dataset = CyberTrendImageDataset(data_dir=CONSTANTS['OUTPUT_DIR'], split='val')
test_dataset = CyberTrendImageDataset(data_dir=CONSTANTS['OUTPUT_DIR'], split='test')

# Initialize DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONSTANTS['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONSTANTS['BATCH_SIZE'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONSTANTS['BATCH_SIZE'], shuffle=False)

print(f"\nDataLoaders ready:")
print(f" - Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f" - Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f" - Test:  {len(test_dataset)} samples, {len(test_loader)} batches")

# Verify sample format
sample = train_dataset[0]
print(f"\nSample format:")
print(f" - Keys: {list(sample.keys())}")
print(f" - Input shape: {sample['input'].shape}")
print(f" - Target shape: {sample['target'].shape}")

## Model Initialization

Initialize VisionTS model using the wrapper class.

In [ ]:
# Get dimensions from dataset
sample = train_dataset[0]
context_len = sample['input'].shape[0]
pred_len = sample['target'].shape[0]
num_features = sample['input'].shape[1]

print(f"Detected dimensions:")
print(f" - context_len: {context_len}")
print(f" - pred_len: {pred_len}")
print(f" - num_features (nvars): {num_features}")

# Create model
model = create_visionts_model(
    config=VISIONTS_CONFIG,
    context_len=context_len,
    pred_len=pred_len,
    num_features=num_features,
    device=str(device),
)

# Count parameters (matching Graph pipeline style)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\n✅ VisionTS Initialized on {device}")
print(f" - Input Window: {context_len}")
print(f" - Output Horizon: {pred_len}")
print(f" - Features: {num_features}")
print(f" - Parameters: {trainable_params:,} trainable / {total_params:,} total")

## Metrics Functions

RSE and RAE metrics matching the B-MTGNN paper (same as Graph pipeline).

In [ ]:
def compute_metrics(all_preds, all_targets):
    """Compute RSE and RAE metrics (same as Graph pipeline)."""
    preds_flat = all_preds.view(-1)
    targets_flat = all_targets.view(-1)
    target_mean = torch.mean(targets_flat)

    # RSE
    numerator_rse = torch.sqrt(torch.sum((preds_flat - targets_flat) ** 2))
    denominator_rse = torch.sqrt(torch.sum((targets_flat - target_mean) ** 2))
    rse = numerator_rse / (denominator_rse + 1e-7)

    # RAE
    numerator_rae = torch.sum(torch.abs(preds_flat - targets_flat))
    denominator_rae = torch.sum(torch.abs(targets_flat - target_mean))
    rae = numerator_rae / (denominator_rae + 1e-7)

    return rse.item(), rae.item()

print("Metrics functions defined.")

## Training Loop

Short training demonstration with validation metrics.

In [ ]:
# Training setup
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
loss_fn = nn.L1Loss()  # MAE loss (same as Graph pipeline)

epochs = 2  # Short run for verification (VisionTS is slower with high-dim data)
print(f"Starting Training ({epochs} epochs)...")
print(f"Note: Training is slow with {num_features} features. This is expected.\n")

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        x = batch['input'].to(device)
        y_true = batch['target'].to(device)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_fn(y_pred, y_true)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    preds_list, targets_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch['input'].to(device)
            y_true = batch['target'].to(device)
            y_pred = model(x)
            preds_list.append(y_pred.cpu())
            targets_list.append(y_true.cpu())

    all_preds = torch.cat(preds_list, dim=0)
    all_targets = torch.cat(targets_list, dim=0)
    rse, rae = compute_metrics(all_preds, all_targets)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val RSE: {rse:.4f} | Val RAE: {rae:.4f}")

## Forecast Visualization

Visualize model predictions on a validation sample.

In [ ]:
# Get a sample prediction
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader))
    x = batch['input'].to(device)
    y_true = batch['target'].to(device)
    y_pred = model(x)

# Extract data for first sample, first feature
# Shapes: (Batch, Time, Features) for VisionTS
sample_idx = 0
feature_idx = 0

history = x[sample_idx, :, feature_idx].cpu().numpy()
ground_truth = y_true[sample_idx, :, feature_idx].cpu().numpy()
forecast = y_pred[sample_idx, :, feature_idx].cpu().numpy()

# Create time axes
t_history = np.arange(0, CONSTANTS['CONTEXT_LEN'])
t_future = np.arange(CONSTANTS['CONTEXT_LEN'], CONSTANTS['CONTEXT_LEN'] + CONSTANTS['PRED_LEN'])

# Plot (matching Graph pipeline style)
plt.figure(figsize=(10, 5))
plt.plot(t_history, history, label=f'History ({CONSTANTS["CONTEXT_LEN"]} Months)',
         marker='o', color='black')
plt.plot(t_future, ground_truth, label=f'Ground Truth ({CONSTANTS["PRED_LEN"]} Months)',
         alpha=0.5, color='gray')
plt.plot(t_future, forecast, label='VisionTS Forecast',
         linestyle='--', marker='x', color='blue')
plt.axvline(x=CONSTANTS['CONTEXT_LEN']-0.5, color='red', linestyle=':', label='Forecast Start')
plt.xlabel('Time (Months)')
plt.ylabel('Value')
plt.title(f"VisionTS Forecast: Feature 0 ({df_raw.columns[feature_idx]})")
plt.legend()
plt.tight_layout()
plt.show()